<a href="https://colab.research.google.com/github/diegozuu/asistente-para-planificacion-de-toma-de-ramos/blob/main/PROTOTIPO%202%20EVALUACION%201/PROTOTIPO%202%20EVALUACION%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LIBRERIAS A USAR

In [ ]:
!pip install pdfplumber gradio openai pandas -q

import pdfplumber
import pandas as pd
import gradio as gr
from openai import OpenAI
from google.colab import userdata

print("librerias importadas correctamente")

librerias importadas correctamente


# COMPROBACION DE FUNCIONAMIENTP DE LA APY KEY Y SELECION DE MODELO A USAR

In [ ]:
try:
    cliente_ai = OpenAI(
        api_key=userdata.get('OPENAI_API_KEY'), #COMPRUEBA QUE TU API KEY ESTE ACTIVA Y COINCIDA CON EL NOMBRE DE LA INGRESADA
        base_url="https://api.groq.com/openai/v1"
    )
    modelo_elegido = "openai/gpt-oss-120b"#MODELO A USAR DE GROUQ
    print("API key y modelo IA funcionado correctamente")
except Exception as e:
    print(f"⚠️ Error al conectar con la API: {e}")


API key y modelo IA funcionado correctamente


# CARGAR BASE DE DATOS DE LAS PROPUESTAS DE LA INSTITUCION EN ESTE CASO EN FORMATO EXCEL

In [ ]:
try:
    df_horarios = pd.read_excel('SAN BERNARDO 2026 002.xlsx', sheet_name='BASE')
    print("✅ Excel de horarios cargado correctamente.")
except Exception as e:
    print(f"⚠️ Error al cargar el Excel: {e}")
    df_horarios = pd.DataFrame()

✅ Excel de horarios cargado correctamente.


# CREACION DE FUNCION LA CUAL PERMITE EXTRAER TEXTO DE LOS PDF

In [ ]:
def extraer_texto_pdf(ruta_pdf):
    texto = ""
    try:
        with pdfplumber.open(ruta_pdf) as pdf:
            for pagina in pdf.pages:
                texto += pagina.extract_text() + "\n"
    except Exception:
        pass
    return texto

# IMPORTACION DE LAS MAYAS A USAR

In [ ]:
try:
  # 1.(Ej: "Mecánica", "Desarrollo"): Son las palabras clave que busca el bot en el chat para identificar la carrera del alumno.
  # 2.Asegúrate de que los nombres de los archivos .pdf coincidan EXACTAMENTE con los nombres de los archivos que subiste a Google Colab.
  # 3."PalabraClave": "NombreExactoDelArchivo.pdf"
  mallas_archivos = {
      "Mecánica": "MALLA_CURRICULAR-Ingeniería-en-Mecánica-Automotriz-y-Autotrónica-DuocUC-2026.pdf",
      "Finanzas": "MALLA_CURRICULAR-Ingeniería-en-Administración-mención-Finanzas-DuocUC-2026.pdf",
      "Automatización": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Automatización-DuocUC-2026.pdf",
      "Ciencia de Datos": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ciencia-de-Datos-DuocUC-2026.pdf",
      "Ingeniería de Desarrollo": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Desarrollo-de-Software-DuocUC-2026.pdf",
      "Ingeniería de Datos": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ingeniería-de-Datos-DuocUC-2026.pdf",
      "Inteligencia Artificial": "MALLA_CURRICULAR-Ingeniería-en-Informática-Mención-Inteligencia-Artificial-DuocUC-2026.pdf"
  }

  base_mallas_txt = {clave: extraer_texto_pdf(arch) for clave, arch in mallas_archivos.items()}
  print("✅ Mallas curriculares procesadas.\n")
except Exception as e:
    print(f"⚠️ Error al almacenar las mayas pdf: {e}")

✅ Mallas curriculares procesadas.



# Creacion de funcion de filtrado dinámico

In [ ]:
def obtener_contexto_especifico(historial_chat_texto):
    texto_malla = ""
    horarios_txt = ""
    if df_horarios.empty: return texto_malla, "No hay datos de horarios."

    for clave in mallas_archivos.keys():
        if clave.lower() in historial_chat_texto.lower():
            texto_malla = base_mallas_txt.get(clave, "")
            filtro = df_horarios['Carrera'].str.contains(clave, case=False, na=False)
            cols = ['Carrera', 'Nombre Asignatura', 'Sección', 'Horario', 'Docente']
            df_sub = df_horarios[filtro][cols]
            horarios_txt = df_sub.to_string(index=False)
            break
    return texto_malla, horarios_txt

#  SE LE ENTREGA UN PROMPT EL CUAL LE DA UN OBJETIVO A LA IA Y TAMBIEN LE EXIGE ENTREGAR DE CIERTA FORMA Y CON CIERTA ESTRUTURA EL HORARIO AL USUARIO

In [ ]:
def responder_chat(mensaje, historial):
    # Extraer todo el texto acumulado (compatible con nuevo formato de Gradio)
    texto_total = mensaje
    for m in historial:
        if isinstance(m, dict):
            texto_total += " " + str(m.get("content", ""))
        elif isinstance(m, (list, tuple)):
            if m[0]: texto_total += " " + str(m[0])
            if m[1]: texto_total += " " + str(m[1])

    malla_filtrada, oferta_filtrada = obtener_contexto_especifico(texto_total)

    prompt_sistema = f"""
Eres el "Asistente Virtual DUOC UC". Tu función es interactuar amablemente y generar propuestas de horarios académicos.

INSTRUCCIONES DE COMPORTAMIENTO:
1. SI EL USUARIO SOLO SALUDA (ej: "Hola", "Buenas tardes", "¿Cómo estás?") O NO MENCIONA SU CARRERA O SEMESTRE:
   - Responde cordialmente el saludo.
   - Preséntate brevemente diciendo que eres el Asistente Virtual DUOC UC.
   - Solicítale amablemente que indique su **carrera**, el **semestre** que va a cursar y su **jornada** (Diurna o Vespertina) para ayudarle a armar su propuesta de horario.
   - NO muestres la tabla de horario aún.

2. SI EL USUARIO TE PIDE GENERAR SU HORARIO O INDICA SU CARRERA / SEMESTRE:
   - Genera OBLIGATORIAMENTE el boletín de horario usando una tabla Markdown.
   - Usa los datos de la malla asignada. Guía para Desarrollo de Software:
     * 3er Semestre: Desarrollo Full Stack I, Base de Datos Aplicada II, Ingeniería de Software, Estadística Descriptiva, Inglés Intermedio I.
     * 4to Semestre: Desarrollo de Aplicaciones Móviles, Desarrollo Full Stack II, Taller de Base de Datos, Ética para el Trabajo, Estadística Descriptiva / Inglés.
   - Formato estricto de la tabla:
     | Módulo Horario | Lunes | Martes | Miércoles | Jueves | Viernes | Sábado |
     |---|---|---|---|---|---|---|
     | 08:30 - 09:50 | NOMBRE DEL RAMO<br>SECCIÓN | | NOMBRE DEL RAMO<br>SECCIÓN | | | |

DATOS EXTRAÍDOS DEL SISTEMA:
- Malla: {malla_filtrada if malla_filtrada else "DATOS NO ENCONTRADOS"}
- Oferta: {oferta_filtrada if oferta_filtrada else "DATOS NO ENCONTRADOS"}
"""

    mensajes_api = [{"role": "system", "content": prompt_sistema}]

    # Adaptar historial para la API
    for m in historial:
        if isinstance(m, dict):
            mensajes_api.append({"role": m.get("role", "user"), "content": m.get("content", "")})
        elif isinstance(m, (list, tuple)):
            if m[0]: mensajes_api.append({"role": "user", "content": m[0]})
            if m[1]: mensajes_api.append({"role": "assistant", "content": m[1]})

    mensajes_api.append({"role": "user", "content": mensaje})

    try:
        respuesta = cliente_ai.chat.completions.create(
            model=modelo_elegido,
            messages=mensajes_api,
            temperature=0.3
        )
        return respuesta.choices[0].message.content
    except Exception as e:
        return f"⚠️ Error de conexión: {e}"

# ASIGNACION DE ESTRUTURA HTML LA CUAL SE USARA

In [ ]:
try:
  estilos_css = """
  :root {
      --duoc-yellow: #FFB81C;
      --duoc-black: #1A1A1A;
      --duoc-white: #FFFFFF;
  }
  body { background-color: #EAECEF !important; font-family: 'Arial', sans-serif !important; }
  .gradio-container {
      border: 5px solid var(--duoc-yellow) !important;
      border-radius: 12px !important;
      background-color: var(--duoc-white) !important;
      box-shadow: 0px 8px 20px rgba(0,0,0,0.1) !important;
      padding: 0 !important; overflow: hidden;
  }
  .encabezado-duoc {
      background-color: var(--duoc-black); color: var(--duoc-white);
      padding: 25px; border-bottom: 5px solid var(--duoc-yellow);
      display: flex; align-items: center; justify-content: center;
      position: relative; margin-bottom: 10px;
  }
  .encabezado-duoc img { height: 60px; position: absolute; left: 30px; background-color: white; padding: 5px; border-radius: 8px; }
  .encabezado-textos { text-align: center; }
  .encabezado-duoc h1 { color: var(--duoc-yellow) !important; margin: 0; font-size: 28px; font-weight: bold; text-transform: uppercase; }
  .encabezado-duoc p { margin: 5px 0 0 0; font-size: 15px; color: #e0e0e0; }
  .sub-encabezado {
      background-color: var(--duoc-black); color: var(--duoc-white);
      padding: 15px; margin: 10px 20px 20px 20px; border-radius: 8px;
      border: 2px solid var(--duoc-yellow); text-align: center; box-shadow: 0px 4px 10px rgba(0,0,0,0.15);
  }
  .sub-encabezado h3 { color: var(--duoc-yellow) !important; margin: 0 0 5px 0; font-size: 20px; }
  .sub-encabezado p { margin: 0; font-size: 14px; color: #CCCCCC; }

  .message.bot, .message.user {
      background-color: #FFF6E0 !important; color: #000000 !important;
      box-shadow: 0px 2px 5px rgba(0,0,0,0.05) !important;
  }
  .message.bot { border-radius: 20px 20px 20px 4px !important; border: none !important; }
  .message.user { border: 2px solid var(--duoc-yellow) !important; border-radius: 20px 20px 4px 20px !important; }
  .message.bot *, .message.user * { color: #000000 !important; }

  .prose table, .prose td, .prose th { background-color: #FFFFFF !important; color: #000000 !important; border-color: #CCCCCC !important; }
  .prose td * { color: #000000 !important; }
  .prose th { background-color: var(--duoc-black) !important; color: var(--duoc-yellow) !important; text-align: center !important; font-weight: bold !important; }
  .prose td { text-align: center !important; vertical-align: middle !important; font-size: 13px !important; padding: 12px !important; }
  footer { background-color: var(--duoc-black) !important; border-top: 5px solid var(--duoc-yellow) !important; color: var(--duoc-white) !important; padding: 10px !important; margin-top: 10px !important; }
  footer a, footer span { color: var(--duoc-yellow) !important; }
  """

  html_cabecera = """
  <div class="encabezado-duoc">
      <img src="https://upload.wikimedia.org/wikipedia/commons/a/aa/Logo_DuocUC.svg" alt="Logo Duoc UC">
      <div class="encabezado-textos">
          <h1>Portal Académico de Estudiantes</h1>
          <p>Simulador de Toma de Ramos y Boletín de Carga Académica</p>
      </div>
  </div>
  <div class="sub-encabezado">
      <h3>🎓 Planificador Virtual de Horarios</h3>
      <p>Indica tu carrera, semestre y jornada para generar tu boletín académico.</p>
  </div>
  """
  print("html correctamente asignado a las variables")
except Exception as e:
  print(f"la estructura no se ha almacennado correctamente popr el error {e}")

html correctamente asignado a las variables


# ⚠️ REQUISITO PREVIO CRÍTICO:
Antes de interactuar con la interfaz, asegúrate de haber ejecutado todas las celdas anteriores y verificar que los archivos de las mallas curriculares (PDFs) y la oferta de horarios (Excel) estén subidos en el almacenamiento de la sesión de Google Colab.


#📌 IMPORTANTE SOBRE LOS NOMBRES DE ARCHIVO:
Los nombres de los archivos subidos a Colab deben coincidir exactamente con los nombres definidos en las celdas de código previas:

El nombre del archivo Excel debe ser idéntico al escrito en la sección de Cargar base de datos (pd.read_excel('NOMBRE_DE_TU_ARCHIVO.xlsx')).

Los nombres de los PDFs de las mallas deben coincidir exactamente con los declarados en el diccionario mallas_archivos.

Si modificas o renombras algún archivo subido, debes actualizar la cadena correspondiente en el código para evitar errores de lectura.

# INICIAR EL FUNCIONAMIENTO DE LA SIMULACION

In [ ]:
mensaje_inicial = [
    {
        "role": "assistant",
        "content": "👋 ¡Hola! Soy tu **Asistente Virtual DUOC UC**.\n\nEstoy aquí para ayudarte a planificar tu horario y generar tu boletín de carga académica. Por favor, dime tu **carrera**, el **semestre** que vas a cursar y tu **jornada** (Diurna o Vespertina) para comenzar."
    }
]

with gr.Blocks() as portal_simulado:
    gr.HTML(html_cabecera)

    chat = gr.ChatInterface(
        fn=responder_chat,
        chatbot=gr.Chatbot(value=mensaje_inicial, height=520, show_label=False),
        textbox=gr.Textbox(placeholder="Ej: Hola, soy de Ingeniería en Informática...", container=False, scale=7)
    )

print("Generando link del html en funcionamiento....")
portal_simulado.launch(share=True, css=estilos_css)

Generando link del html en funcionamiento....
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9cfc6f3937c791c121.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#PROMTP DE PRUBEA
Hola, soy estudiante de Ingeniería en Informática mención Desarrollo de Software, voy a pasar a 4to semestre en jornada Diurna. ¿Me generas la propuesta de horario por favor?